In [ ]:
#!/usr/bin/env python3
"""
v5 G0 — VCC leg diagnostics. Formalizes the checks that overturned the
auto-generated INVALID verdict in `results/v5 G0 - VCC/v5g0_vcc_report_v1.json`.
Engineer, 2026-07-19, per Brian's ruling of 2026-07-19 (loss of the noise-model
arithmetic below appears in his handoff; this script reproduces the raw checks
that fed it).

BACKGROUND
  The auto-generated report called global reproducibility 0.048 (< 0.80 gate)
  INVALID = "pipeline bug, not biology." Three checks below show that is wrong:
  the pipeline is clean and P-power is not the cause. What is left is real:
  the 150 VCC perturbations do not share a detectable global program (gamma ~ 0
  under Brian's noise model corr(g^A,g^B) = |gamma|^2 / (|gamma|^2 + N/P)), so
  d_p already IS (approximately) the specific component. See
  docs/HANDOFF_v5_G0_VCC_real_data_to_brian.md for the full ruling.

INPUT
  results/v5 G0 - VCC/v5g0_vcc_accum_v1.npz
  Bin layout: bid(group, half, slot) = (group*n_half + half)*n_slot + slot,
  contiguous per (group, half) -> depth n is a prefix sum over slots 0..k.
  depths = [25, 50, 100, 200, 400, 600]; slot 6 (the 7th) is the REMAINDER bin,
  i.e. summing all 7 slots gives that group's full available half-depth. n_half
  = 3: halves 0/1 are the A/B reproducibility split, half 2 is the held-out
  panel C used only for effect-size stratification (never for measurement, to
  avoid the circular-selection bug flagged 2026-07-18).

CHECKS
  1. P-power: does enlarging P (44 -> 148) at fixed depth move global corr?
     (No — ruled out my own P-under-averaging hypothesis.)
  2. Pipeline: control pseudobulk, half A vs half B, no perturbation involved.
     (r = 0.999 @ n=600 -> pipeline is clean.)
  3. Heterogeneity: per-perturbation d_p^A vs d_p^B among the best-powered
     targets, same depth. (4x spread -> real, not power.)
"""
import os

import numpy as np

# Two supported layouts:
#   sandbox (default) - reads straight from the repo's results/ folder.
#   Kaggle - set V5G0_VCC_ACCUM, or attach a Dataset containing
#     v5g0_vcc_accum_v1.npz and it resolves at
#     /kaggle/input/datasets/<username>/<dataset-slug>/v5g0_vcc_accum_v1.npz
#     (same convention as the VCC probe run and the Replogle notebook).
ROOT = "/sessions/peaceful-cool-bohr/mnt/Project 13"
_SANDBOX_DEFAULT = os.path.join(ROOT, "results", "v5 G0 - VCC", "v5g0_vcc_accum_v1.npz")
_KAGGLE_DEFAULT = ("/kaggle/input/datasets/ericdu847/v5g0-vcc-00-probe/"
                   "v5g0_vcc_accum_v1.npz")
ACCUM = os.environ.get(
    "V5G0_VCC_ACCUM",
    _KAGGLE_DEFAULT if os.path.isdir("/kaggle/input") else _SANDBOX_DEFAULT)


def load():
    z = np.load(ACCUM, allow_pickle=True)
    return z


def bid(gi, h, s, n_half, n_slot):
    return (gi * n_half + h) * n_slot + s


def corr(a, b):
    a = a - a.mean()
    b = b - b.mean()
    d = np.sqrt((a * a).sum() * (b * b).sum())
    return float((a * b).sum() / d) if d > 0 else np.nan


def pseudobulk(A, gi, half, slot, gm, n_half, n_slot):
    s = A[bid(gi, half, 0, n_half, n_slot):bid(gi, half, slot, n_half, n_slot) + 1].sum(axis=0)
    s = s[gm]
    tot = s.sum()
    return np.log1p(s / tot * 1e6) if tot > 0 else None


def run():
    z = load()
    accum, groups, avail = z["accum"], list(z["groups"]), z["avail"]
    run_kind, depths, detect = list(z["run_kind"]), list(z["depths"]), z["detect"]
    n_group, n_half, n_slot = int(z["n_group"]), int(z["n_half"]), int(z["n_slot"])
    n_cells = int(z["n_cells"])
    ctrl = groups.index("non-targeting")
    gm = (detect / n_cells) >= 0.01
    r0 = run_kind.index("real")
    A = accum[r0]

    print("=== CHECK 1: P-power. Does enlarging P move global corr at fixed depth? ===")
    for depth_i, depth in enumerate(depths):
        qual = [gi for gi in range(n_group) if gi != ctrl
               and avail[r0][gi, 0] >= depth and avail[r0][gi, 1] >= depth]
        matched44 = [gi for gi in range(n_group) if gi != ctrl
                    and avail[r0][gi, 0] >= 600 and avail[r0][gi, 1] >= 600]

        def g_of(pool):
            cA = pseudobulk(A, ctrl, 0, depth_i, gm, n_half, n_slot)
            cB = pseudobulk(A, ctrl, 1, depth_i, gm, n_half, n_slot)
            dA = np.mean([pseudobulk(A, gi, 0, depth_i, gm, n_half, n_slot) - cA for gi in pool], axis=0)
            dB = np.mean([pseudobulk(A, gi, 1, depth_i, gm, n_half, n_slot) - cB for gi in pool], axis=0)
            return corr(dA, dB)

        r_full, r_44 = g_of(qual), g_of(matched44)
        print(f"  depth={depth:4d}  P_full={len(qual):3d} global_r={r_full:+.4f}  |  "
              f"P=44 global_r={r_44:+.4f}")
    print("  -> P barely moves the number. Ruled out as the cause of low global corr.\n")

    print("=== CHECK 2: pipeline sanity — control vs control, no perturbation ===")
    for depth_i, depth in enumerate(depths):
        cA = pseudobulk(A, ctrl, 0, depth_i, gm, n_half, n_slot)
        cB = pseudobulk(A, ctrl, 1, depth_i, gm, n_half, n_slot)
        print(f"  depth={depth:4d}  corr(ctrl_A, ctrl_B) = {corr(cA, cB):+.4f}")
    print("  -> approaches 0.999 by n=600. Pipeline is clean; INVALID is not a bug.\n")

    print("=== CHECK 3: heterogeneity among best-powered perturbations, n=600 ===")
    qual600 = [gi for gi in range(n_group) if gi != ctrl
              and avail[r0][gi, 0] >= 600 and avail[r0][gi, 1] >= 600]
    biggest = sorted(qual600, key=lambda gi: -avail[r0][gi, 0])[:5]
    slot600 = len(depths) - 1
    cA = pseudobulk(A, ctrl, 0, slot600, gm, n_half, n_slot)
    cB = pseudobulk(A, ctrl, 1, slot600, gm, n_half, n_slot)
    for gi in biggest:
        pA = pseudobulk(A, gi, 0, slot600, gm, n_half, n_slot) - cA
        pB = pseudobulk(A, gi, 1, slot600, gm, n_half, n_slot) - cB
        print(f"  {groups[gi]:10s} avail={avail[r0][gi,0]:5d}  corr(d_p^A,d_p^B) @ n=600 = {corr(pA,pB):+.4f}")
    print("  -> 4x spread among the best-powered targets. Real, not a power artifact.")


if __name__ == "__main__":
    run()